# 05. Temporal tiling: первый кадр, lattice и context

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Выведем temporal coordinate map и проверим длины/значения оригинальных encode/decode tile loops на известном временном сигнале.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT предпосылка:** Lab 3 Q4.3/Q4.5 объясняют down/up sampling; переход от image к temporal video — дополнительная тема. Frame lattice и temporal context не следует приписывать исходному MNIST заданию. [Мост image→video](../MIT_LAB3_BRIDGE.md).


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='05_temporal_tiling_and_first_frame';metrics={}
print('Pinned source:',source_hash())


## 1. Temporal lattice

При F=4k+1 encoder выдаёт k+1 latents с nominal anchor frames 0,4,8,…,4k. Это lattice, а не утверждение, что latent зависит только от одного кадра. Conv history и normalization расширяют зависимость.

В исходнике sample tile содержит Q+1 кадров; его latent length Q/4+1. Начала temporal tiles идут с шагом S кадров, кратным 4. Для tiles после первого удаляется первый latent; decoder аналогично удаляет первый **sample frame**:

$$
F_z=1+(F-1)/4,\qquad Q_z=Q/4,\quad S_z=S/4,\quad O_z=(Q-S)/4.
$$

Source defaults Q=16,S=12; adaptive planner ставит S=8. Это разные policies. `apply_tiling` принимает ft=Q+1 и хранит min_num_frames=ft−1. Забыть −1 означает сдвинуть lattice.


In [ ]:
show_source('_temporal_tiled_encode');show_source('_temporal_tiled_decode');show_source('blend_t')
obj=oracle_codec();F=33
x=torch.arange(F,dtype=torch.float64).reshape(1,1,F,1,1).expand(1,1,F,8,8)
with torch.inference_mode():
    full=obj.encoder(x);encoded=obj._temporal_tiled_encode(x)
    # spatial tiling disabled: H/W smaller than spatial tile min.
    decoded=obj._temporal_tiled_decode(full).sample
    expected_dec=obj.decoder(full)
assert torch.equal(full,encoded) and torch.equal(decoded,expected_dec)
metrics['aligned_temporal_oracle']={'F':F,'Fz':encoded.shape[2],'encode_max_error':float((full-encoded).abs().max()),'decode_max_error':float((decoded-expected_dec).abs().max())}
print('latent time anchors:',encoded[0,0,:,0,0].tolist())


## 2. Frame-count audit: F=4k+1 ещё недостаточно

Если last start не доходит до конца, temporal loop теряет tail; первый tile отдельно обрезается до S_z+1. Поэтому нельзя переносить geometry encoder на geometry stitching автоматически.

Будем вызывать `_encode`, то есть **реальный dispatch**, а не temporal private method на коротком input, для которого он не предназначен. Для F≤Q+1 пойдёт full encode. Для длинных F сравним original tile output с expected lattice. Shape failure — результат опыта, а не повод допадить output незаметно.


In [ ]:
rows=[]
for frames in [1,9,17,21,25,29,33,37,41,49]:
    obj=oracle_codec();obj.use_tiling=False
    x=torch.arange(frames,dtype=torch.float64).reshape(1,1,frames,1,1).expand(1,1,frames,8,8)
    out=obj._encode(x);expected=obj.encoder(x)
    rows.append({'frames':frames,'expected_Fz':expected.shape[2],'actual_Fz':out.shape[2],
                 'same_shape':out.shape==expected.shape})
assert any(not r['same_shape'] for r in rows)
metrics['temporal_encode_lengths']=rows
for row in rows:print(row)
fig,ax=plt.subplots(figsize=(7,3));ax.plot([r['frames'] for r in rows],[r['expected_Fz'] for r in rows],'o-',label='Expected lattice');ax.plot([r['frames'] for r in rows],[r['actual_Fz'] for r in rows],'x--',label='Original stitch')
ax.set(xlabel='Input F',ylabel='Latent Fz',title='Original adaptation: temporal coverage audit');ax.legend();fig.tight_layout();fig.savefig(figure_path(LAB,'length_audit'));plt.show()


## 3. Проверяем decoder отдельно

Encoder и decoder loops не обязаны ошибаться на одних и тех же lengths. Decode принимает F_z, а temporal tile slice имеет Q_z+1. На выходе first frame special rule даёт 4(F_z−1)+1. Сравним ожидаемую и фактическую длину **для каждого пути отдельно**.

Это unit evidence о pinned algorithm. Возможная защитная обвязка в production — дополнить последовательность до поддерживаемой grid, выполнить encode/decode, затем явно обрезать original length, записав valid-frame mask. Нельзя без доказательства считать padding нейтральным: GroupNorm изменяет все значения.


In [ ]:
rows=[]
for fz in [1,3,5,6,7,8,9,10,11,13]:
    obj=oracle_codec();obj.use_tiling=False
    z=torch.arange(fz,dtype=torch.float64).reshape(1,1,fz,1,1)
    out=obj._decode(z).sample
    rows.append({'Fz':fz,'expected_frames':4*(fz-1)+1,'actual_frames':out.shape[2]})
metrics['temporal_decode_lengths']=rows
for row in rows:print(row)


## 4. Reset history и временные швы

Каждый temporal tile заново делает replicate-padding своего первого кадра. Для causal conv receptive history h кадров предыдущего tile нужна как prefix, а не как новый «начальный момент мира». Можно переносить conv cache или передавать halo и отбрасывать невалидные outputs. Но GroupNorm и attention снова мешают exact streaming equivalence.

Измерим full vs temporal на длине с правильным coverage, чтобы не смешивать shape failure с approximate context. Random tiny encoder/decoder, без optional attention. Граф показывает mean squared difference по latent time; первый latent не обязан быть единственным изменившимся.


In [ ]:
codec=TinyCodec(seed=2);x=synthetic_video(33,32,32)
with torch.inference_mode():
    full=codec._encode(x)
    codec.set_tiles(spatial=False,temporal=True)
    tiled=codec._encode(x)
stat=errors(full,tiled);metrics['native_tiny_temporal_difference']=stat
profile=(full-tiled).square().mean((0,1,3,4)).numpy()
fig,ax=plt.subplots(figsize=(7,3));ax.plot(profile,'o-');ax.set(xlabel='Latent temporal index',ylabel='Squared difference',title='Same random encoder: full vs temporal')
fig.tight_layout();fig.savefig(figure_path(LAB,'temporal_error'));plt.show();print(stat)


## Упражнения и критерий перехода

1. Выпишите все tile slices и dropped elements для F=33,Q=16,S=8.
2. Повторите oracle length sweep с S=12. Найдите условия отсутствия tail loss.
3. Сравните input replication и conv-cache подход на простом 1D causal filter. Переход: объясните отдельно timestamp, context, coverage и sample count.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
